In [29]:
from pinecone import Pinecone,ServerlessSpec
from sentence_transformers import SentenceTransformer
from google import genai 
import time
import pandas as pd
import os
import dotenv
dotenv.load_dotenv()


True

In [5]:
# Load variable
model_name = os.getenv("MODEL_NAME")
pinecone_api_key = os.getenv("PINECONE_API_KEY")
index_name = os.getenv("PINECONE_INDEX_NAME")

In [7]:
# Load model (downloads automatically on first run)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Initialize pinecone
pc = Pinecone(api_key=pinecone_api_key)

# Try out embeddings

In [9]:
embeddings = model.encode("Hello World")
embeddings

array([-3.44772860e-02,  3.10231932e-02,  6.73497282e-03,  2.61090156e-02,
       -3.93620320e-02, -1.60302490e-01,  6.69239759e-02, -6.44144695e-03,
       -4.74505089e-02,  1.47588737e-02,  7.08753616e-02,  5.55275753e-02,
        1.91932973e-02, -2.62513310e-02, -1.01095187e-02, -2.69405209e-02,
        2.23074257e-02, -2.22266577e-02, -1.49692655e-01, -1.74930580e-02,
        7.67622096e-03,  5.43523021e-02,  3.25445435e-03,  3.17259841e-02,
       -8.46214145e-02, -2.94059832e-02,  5.15956655e-02,  4.81240191e-02,
       -3.31475982e-03, -5.82791939e-02,  4.19692770e-02,  2.22107302e-02,
        1.28188819e-01, -2.23389044e-02, -1.16562909e-02,  6.29283711e-02,
       -3.28762941e-02, -9.12260711e-02, -3.11752949e-02,  5.26995696e-02,
        4.70348299e-02, -8.42030495e-02, -3.00561879e-02, -2.07448397e-02,
        9.51781031e-03, -3.72175407e-03,  7.34332670e-03,  3.93243395e-02,
        9.32740718e-02, -3.78859066e-03, -5.27420826e-02, -5.80581762e-02,
       -6.86437730e-03,  

In [10]:
len(embeddings)

384

# Wrangle Dataset

In [11]:
df = pd.read_json("products/products.jsonl",lines=True)

In [12]:
df.head(2)

,name,category,description,ingredients,price,rating,image_path
0,Cappuccino,Coffee,A rich and creamy cappuccino made with freshly...,"[Espresso, Steamed Milk, Milk Foam]",4.50,4.7,cappuccino.jpg
1,Jumbo Savory Scone,Bakery,"Deliciously flaky and buttery, this jumbo savo...","[Flour, Butter, Cheese, Herbs, Baking Powder, ...",3.25,4.3,SavoryScone.webp


In [13]:
df['text'] = df['name']+" : "+df["description"]+\
        " -- Ingredients: "+df["ingredients"].astype(str) +\
        " -- Price: "+ df["price"].astype(str) +\
        " -- rating: "+ df["rating"].astype(str)

In [14]:
df["text"].head(2)

0    Cappuccino : A rich and creamy cappuccino made...
1    Jumbo Savory Scone : Deliciously flaky and but...
Name: text, dtype: object

In [15]:
texts = df["text"].tolist()

In [16]:
texts[:2]

["Cappuccino : A rich and creamy cappuccino made with freshly brewed espresso, steamed milk, and a frothy milk cap. This delightful drink offers a perfect balance of bold coffee flavor and smooth milk, making it an ideal companion for relaxing mornings or lively conversations. -- Ingredients: ['Espresso', 'Steamed Milk', 'Milk Foam'] -- Price: 4.5 -- rating: 4.7",
 "Jumbo Savory Scone : Deliciously flaky and buttery, this jumbo savory scone is filled with herbs and cheese, creating a mouthwatering experience. Perfect for a hearty snack or a light lunch, it pairs beautifully with your favorite coffee or tea. -- Ingredients: ['Flour', 'Butter', 'Cheese', 'Herbs', 'Baking Powder', 'Salt'] -- Price: 3.25 -- rating: 4.3"]

In [ ]:
with open("products/Merry's_way_about_us.txt") as f:
    Merry_way_about_section = f.read()

Merry_way_about_section = "Coffee shop Merry's Way about section: "+ Merry_way_about_section
texts.append(Merry_way_about_section)

In [19]:
with open("products/menu_items_text.txt") as f:
    menu_items_text = f.read()

menu_items_text = "Menu Items: "+ menu_items_text
texts.append(menu_items_text)

In [20]:
menu_items_text

'Menu Items: Menu Items\n\nCappuccino - $4.50\nJumbo Savory Scone - $3.25\nLatte - $4.75\nChocolate Chip Biscotti - $2.50\nEspresso shot - $2.00\nHazelnut Biscotti - $2.75\nChocolate Croissant - $3.75\nDark chocolate (Drinking Chocolate) - $5.00\nCranberry Scone - $3.50\nCroissant - $3.25\nAlmond Croissant - $4.00\nGinger Biscotti - $2.50\nOatmeal Scone - $3.25\nGinger Scone - $3.50\nChocolate syrup - $1.50\nHazelnut syrup - $1.50\nCarmel syrup - $1.50\nSugar Free Vanilla syrup - $1.50\nDark chocolate (Packaged Chocolate) - $3.00'

# Generate Embeddings

In [27]:
embeddings = model.encode(texts)

In [28]:
len(embeddings)

20

In [25]:
embeddings[0] # Embeddings is a 2D - array

array([ 3.35254222e-02, -5.78995384e-02, -1.78688182e-03,  5.48821315e-02,
        5.80844693e-02, -2.60436267e-04,  1.67872421e-02,  4.22078855e-02,
       -2.24657897e-02, -8.33211641e-04, -8.28016177e-03, -9.29772407e-02,
       -1.44812586e-02, -1.98469386e-02, -1.56854410e-02, -7.40084648e-02,
        2.65764575e-02, -2.82377028e-03,  9.70730409e-02,  2.02372428e-02,
        6.77295495e-03, -5.52883893e-02,  3.20757329e-02,  1.05686314e-01,
        1.07744038e-01,  9.20844078e-02,  5.01842052e-02,  2.53406409e-02,
       -4.62080017e-02, -7.17794374e-02, -5.21970652e-02, -6.34094179e-02,
        6.74789026e-02, -2.80871987e-02, -6.89997748e-02,  8.34889039e-02,
        1.01793841e-01, -4.04247083e-02,  5.78652807e-02,  8.21567723e-04,
       -1.02939541e-02,  3.71070579e-02,  9.35883522e-02,  2.11805459e-02,
        7.27651641e-02,  2.55678594e-02, -4.42702323e-02,  1.48766767e-02,
        9.29408427e-03,  5.60665876e-02, -7.44238198e-02, -1.10756278e-01,
        1.04843061e-02,  

# Push data to database

In [30]:
pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    )
)

{
    "name": "coffeeshop",
    "metric": "cosine",
    "host": "coffeeshop-ubukl4o.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [32]:
# wait for the index to be ready
while not pc.describe_index(index_name).status.ready:
    time.sleep(1)

index = pc.Index(index_name)

vectors = []
for text,e in zip(texts,embeddings):
    entry_id = text.split(":")[0]
    vectors.append({
        "id": entry_id,
        "values": e,
        "metadata":{"text":text}
    })

index.upsert(vectors=vectors,
             namespace='ns1'
             )

{'upserted_count': 20}

# Get Closest Documents